# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | khaleelalaqel| 
| **Mehmet Kilinc 25135007** | ** |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [7]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




------------

### 2.4 Evaluatiefuncties
Deze functies gebruiken we bij alle modellen:

- `get_scores()` geeft per patiënt een score: de kans op stroke (`predict_proba`) of, bij een SVM, de afstand tot het hypervlak (`decision_function`).
- `find_best_threshold()` zoekt de **beslisdrempel** met de hoogste F1-score. Een model voorspelt `stroke = 1` als de score boven die drempel ligt. Waarom dit belangrijk is, staat in paragraaf 5.1.
- `F1_BEST_SCORER` is de score die `GridSearchCV` gebruikt: de F1-score bij de beste drempel.
- `metrics_from_confusion()` rekent de metrics **zelf** uit met de formules uit paragraaf 4.2, zodat we kunnen laten zien hoe ze worden berekend.

In [ ]:
def get_scores(model, features):
    """Geeft per patiënt een score: kans op stroke of decision function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def f1_per_threshold(y_true, scores):
    """Berekent de F1-score voor iedere mogelijke beslisdrempel."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    # Het laatste punt van de curve hoort niet bij een drempel, want P=1 , R=0 en geen threshold. 
    f1_values = (2 * precision[:-1] * recall[:-1]
                 / np.maximum(precision[:-1] + recall[:-1], 1e-12))
    return thresholds, f1_values


def find_best_threshold(y_true, scores):
    """Geeft de drempel met de hoogste F1-score en die F1-score."""
    thresholds, f1_values = f1_per_threshold(y_true, scores)
    best = np.argmax(f1_values)
    return thresholds[best], f1_values[best]


def f1_at_best_threshold(y_true, scores):
    """Scorefunctie voor GridSearchCV: F1 bij de beste drempel."""
    return find_best_threshold(y_true, scores)[1]


F1_BEST_SCORER = make_scorer(
    f1_at_best_threshold,
    response_method=("predict_proba", "decision_function"))


def metrics_from_confusion(tn, fp, fn, tp):
    """Berekent de metrics zelf uit de vier cellen van de matrix."""
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    specificity = tn / (tn + fp) if tn + fp > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)
    return {"accuracy": accuracy, "precision": precision,
            "recall": recall, "specificity": specificity, "f1": f1}


def evaluate_predictions(y_true, y_pred, scores):
    """Combineert de metrics uit de confusion matrix met ROC- en PR-AUC."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    results = metrics_from_confusion(tn, fp, fn, tp)
    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"] = average_precision_score(y_true, scores)
    results.update({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
    return results

### 2.5 Klassen `ModelExperiment` en `ResultsTable`
Ieder model wordt op **precies dezelfde manier** getraind, getest en beoordeeld. Daarvoor gebruiken we de klasse `ModelExperiment`. Een model doorloopt vier vaste stappen: hyperparameters tunen met 5-voudige gestratificeerde cross-validation, de beslisdrempel kiezen, testen op de validatieset (die tijdens het tunen niet is gebruikt) en een Kaggle-bestand maken. Doordat elk model dezelfde stappen volgt, kunnen we de resultaten eerlijk vergelijken. De klasse `ResultsTable` verzamelt de resultaten van alle modellen voor de eindtabel in hoofdstuk 6.

| Methode | Wat doet het? |
|---|---|
| `tune()` | **Stap 1:** zoekt de beste hyperparameters met `GridSearchCV` en 5-voudige gestratificeerde cross-validation op de trainingsset. |
| `tune_threshold()` | **Stap 2:** kiest met *out-of-fold* voorspellingen de beslisdrempel met de hoogste F1. |
| `evaluate()` | **Stap 3:** test het getunede model op de validatieset. |
| `make_submission()` | **Stap 4:** traint het model opnieuw op **alle** trainingsdata en maakt een csv-bestand voor Kaggle. |
| `run()` | Voert stap 1 t/m 4 achter elkaar uit en print de resultaten. |
| `best_params_text()` | Geeft de beste hyperparameters als korte, leesbare tekst (zonder `model__` ervoor). |
| `plot_evaluation()` | Toont de confusion matrix op de validatieset. |
| `summary()` | Geeft de resultaten als één rij voor de eindtabel. |
| `ResultsTable.add()` | Voegt een uitgevoerd model toe aan de tabel. |
| `ResultsTable.to_frame()` | Maakt de eindtabel met één rij per model. |


In [ ]:
class ModelExperiment:
    """Doorloopt voor één model de vier vaste stappen.

    name: naam van het model, pipeline: scaler/sampler + model,
    param_grid: hyperparameters voor GridSearchCV,
    techniques: technieken tegen de imbalance (tekst voor de eindtabel).
    """

    def __init__(self, name, pipeline, param_grid, techniques):
        self.name = name
        self.pipeline = pipeline
        self.param_grid = param_grid
        self.techniques = techniques
        self.cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                  random_state=RANDOM_STATE)

    def tune(self, X, y):
        """Stap 1: beste hyperparameters zoeken met GridSearchCV."""
        start = time.perf_counter()
        self.search = GridSearchCV(self.pipeline, self.param_grid,
                                   scoring=F1_BEST_SCORER, cv=self.cv,
                                   n_jobs=N_JOBS)
        self.search.fit(X, y)
        self.tuning_seconds = time.perf_counter() - start
        self.best_model = self.search.best_estimator_

    def tune_threshold(self, X, y):
        """Stap 2: drempel kiezen met de hoogste out-of-fold F1."""
        if hasattr(self.best_model, "predict_proba"):
            method = "predict_proba"
        else:
            method = "decision_function"
        scores = cross_val_predict(clone(self.best_model), X, y,
                                   cv=self.cv, method=method, n_jobs=N_JOBS)
        if method == "predict_proba":
            scores = scores[:, 1]  # alleen de kans op stroke
        self.threshold, self.oof_f1 = find_best_threshold(y, scores)

    def evaluate(self, X, y):
        """Stap 3: het model testen op de validatieset."""
        scores = get_scores(self.best_model, X)
        self.val_pred = (scores >= self.threshold).astype(int)
        self.val_metrics = evaluate_predictions(y, self.val_pred, scores)
        # Ter vergelijking: F1 met de standaarddrempel (0,5 of 0)
        default = 0.5 if hasattr(self.best_model, "predict_proba") else 0.0
        self.val_metrics["f1_default"] = f1_score(
            y, (scores >= default).astype(int))

    def make_submission(self, X, y, X_test, test_ids):
        """Stap 4: opnieuw trainen op alle data en een csv maken."""
        final_model = clone(self.best_model).fit(X, y)
        scores = get_scores(final_model, X_test)
        predictions = (scores >= self.threshold).astype(int)
        file_name = "submission_" + self.name.lower().replace(" ", "_") + ".csv"
        self.submission_path = SUBMISSION_DIR / file_name
        pd.DataFrame({"id": test_ids, "stroke": predictions}).to_csv(
            self.submission_path, index=False)

    def run(self, data):
        """Voert stap 1 t/m 4 uit en print de resultaten."""
        self.tune(data["X_train"], data["y_train"])
        self.tune_threshold(data["X_train"], data["y_train"])
        self.evaluate(data["X_val"], data["y_val"])
        self.make_submission(data["X_full"], data["y_full"],
                             data["X_test"], data["test_ids"])
        m = self.val_metrics
        print(f"Model: {self.name}")
        print(f"Beste hyperparameters: {self.best_params_text()}")
        print(f"CV-F1: {self.search.best_score_:.3f}")
        print(f"Gekozen drempel: {self.threshold:.3f}")
        print(f"Validatie-F1: {m['f1']:.3f} met gekozen drempel, "
              f"{m['f1_default']:.3f} met standaarddrempel")
        print(f"Precision: {m['precision']:.3f}, recall: {m['recall']:.3f}")
        print(f"Tuningtijd: {self.tuning_seconds:.0f} s | "
              f"submission: {self.submission_path}")
        return self

    def best_params_text(self):
        """Beste hyperparameters als tekst, zonder 'model__' ervoor."""
        return ", ".join(f"{key.split('__')[-1]}={value}"
                         for key, value in self.search.best_params_.items())

    def plot_evaluation(self, y_val):
        """Toont de confusion matrix op de validatieset."""
        ConfusionMatrixDisplay.from_predictions(
            y_val, self.val_pred, display_labels=["Geen stroke", "Stroke"],
            cmap="Blues", colorbar=False)
        plt.title(f"{self.name}: validatieset (drempel = {self.threshold:.3f})")
        plt.xlabel("Voorspeld")
        plt.ylabel("Werkelijk")
        plt.grid(False)
        plt.show()

    def summary(self):
        """Resultaten als één rij voor de eindtabel."""
        m = self.val_metrics
        return {"model": self.name,
                "technieken imbalance": self.techniques,
                "CV-F1": self.search.best_score_,
                "validatie-F1": m["f1"],
                "precision": m["precision"],
                "recall": m["recall"],
                "ROC-AUC": m["roc_auc"],
                "PR-AUC": m["pr_auc"],
                "drempel": self.threshold,
                "beste hyperparameters": self.best_params_text()}


class ResultsTable:
    """Verzamelt de resultaten van alle modellen in één tabel."""

    def __init__(self):
        self.experiments = {}

    def add(self, experiment):
        """Voegt een model toe (opnieuw draaien overschrijft de oude rij)."""
        self.experiments[experiment.name] = experiment

    def to_frame(self, kaggle_scores=None):
        """Maakt een DataFrame met één rij per model."""
        rows = [exp.summary() for exp in self.experiments.values()]
        table = pd.DataFrame(rows).set_index("model")
        if kaggle_scores is not None:
            table.insert(2, "Kaggle-F1", pd.Series(kaggle_scores))
        return table

## 4. Opdracht 2 – Evaluatie

### 4.1 Train- en validatieset
We splitsen `train.csv` in een **trainingsset (80%)** en een **validatieset (20%)**. De split is **gestratificeerd**: in beide delen zit hetzelfde percentage stroke (1,54%). Zonder stratificatie kan de kleine groep stroke-patiënten toevallig scheef verdeeld raken.

- Met de **trainingsset** zoeken we hyperparameters en de beslisdrempel (met cross-validation).
- De **validatieset** gebruiken we pas aan het einde, als eerlijke test op data die het model nooit heeft gezien. Deze score noemen we in de eindtabel de *notebookscore*.
- Voor Kaggle trainen we het definitieve model op **alle** 33.544 patiënten.

In [1]:

X_train, X_val, y_train, y_val = dataset.split(test_size=0.2)
X_full, y_full = dataset.get_features_and_target()

# Alle splits in één dict, zodat ieder experiment dezelfde data krijgt
data = {"X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_full": X_full, "y_full": y_full,
        "X_test": dataset.test, "test_ids": dataset.test_ids}

split_overview = pd.DataFrame({
    "aantal patiënten": [len(y_train), len(y_val)],
    "aantal stroke": [y_train.sum(), y_val.sum()],
    "stroke (%)": [y_train.mean() * 100, y_val.mean() * 100]},
    index=["trainingsset", "validatieset"])
split_overview.round(2)

NameError: name 'dataset' is not defined

## 4.2 Metrics voor binaire classificatie

In deze casus is **positief (1) = stroke** en **negatief (0) = geen stroke**. Slechts ongeveer 1,5% van de patiënten heeft een stroke. Daarom zegt één getal zoals accuracy weinig; we kijken naar meerdere metrics.

### Confusion matrix

|                        | **Voorspeld 0 (geen stroke)** | **Voorspeld 1 (stroke)** |
|------------------------|-------------------------------|--------------------------|
| **Werkelijk 0 (geen stroke)** | **TN** (True Negative): gezonde patiënt terecht gerustgesteld | **FP** (False Positive): vals alarm, gezonde patiënt onnodig doorverwezen |
| **Werkelijk 1 (stroke)**      | **FN** (False Negative): gemiste patiënt, stroke niet opgemerkt | **TP** (True Positive): stroke-patiënt terecht opgespoord |

In deze casus is een **FN** het ergst: een gemiste patiënt kan ernstige gevolgen hebben. Een **FP** kost vooral extra onderzoek en onrust.

### Overzicht van de metrics

| Metric | Formule | Betekenis in deze casus |
|--------|---------|-------------------------|
| **Accuracy** | $\dfrac{TP + TN}{TP + TN + FP + FN}$ | Het deel van alle patiënten dat het model goed indeelt. Misleidend hier: een model dat iedereen "geen stroke" geeft, haalt al ongeveer 98,5%. |
| **Precision** | $\dfrac{TP}{TP + FP}$ | Van alle patiënten die het model als risico aanwijst, het deel dat echt een stroke krijgt. Lage precision betekent veel vals alarm. |
| **Recall (sensitiviteit)** | $\dfrac{TP}{TP + FN}$ | Van alle patiënten met een stroke, het deel dat het model vindt. Lage recall betekent veel gemiste patiënten. |
| **Specificiteit** | $\dfrac{TN}{TN + FP}$ | Van alle patiënten zonder stroke, het deel dat het model terecht geruststelt. Door de vele gezonde patiënten is deze bijna altijd hoog. |
| **F1-score** | $2 \cdot \dfrac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ | Eén getal dat precision en recall combineert. Is alleen hoog als het model zowel weinig patiënten mist als weinig vals alarm geeft. |
| **ROC-AUC** | $\displaystyle\int_0^1 \text{TPR} \; d(\text{FPR})$ | De kans dat het model een willekeurige stroke-patiënt een hoger risico geeft dan een willekeurige gezonde patiënt. 0,5 is gokken, 1 is perfect. |
| **PR-AUC** | $\displaystyle\int_0^1 \text{Precision} \; d(\text{Recall})$ | Laat zien hoe betrouwbaar de waarschuwingen blijven als we meer stroke-patiënten willen vinden. Gokken geeft hier ongeveer 0,015 (het aandeel stroke), dus elke stijging daarboven is echte winst. |

*Hierbij is TPR = recall en FPR = $\dfrac{FP}{FP + TN}$ = 1 − specificiteit.*

Accuracy, precision, recall, specificiteit en F1 hangen af van de gekozen beslisdrempel (bijvoorbeeld 0,5), terwijl ROC-AUC en PR-AUC het model over alle drempels heen beoordelen. Bij sterk imbalanced data is PR-AUC informatiever dan ROC-AUC, omdat ROC-AUC door het grote aantal terecht negatieven (TN) optimistisch hoog kan uitvallen, terwijl PR-AUC zich richt op de zeldzame positieve klasse en vals alarm direct zichtbaar maakt (Saito & Rehmsmeier, 2015, *PLoS ONE*).

(Claude, 2026. Prompt 1: &lt; https://claude.ai/share/f7ba8829-bae3-4e66-8cb7-176387434a36 &gt;)

### 4.3 Demonstratie met een zelfgekozen algoritme
We trainen een eenvoudige **logistic regression zonder technieken tegen de imbalance** en met de standaarddrempel 0,5. Daarna rekenen we de metrics zelf uit met `metrics_from_confusion()` en controleren we ze met de functies van scikit-learn.

In [ ]:
baseline = Pipeline([("scaler", StandardScaler()),
                     ("model", LogisticRegression(max_iter=1000))])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)          # drempel 0,5
baseline_scores = get_scores(baseline, X_val)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val, baseline_pred, display_labels=["Geen stroke", "Stroke"],
    cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Baseline: logistic regression, drempel 0,5")
ax.set(xlabel="Voorspeld", ylabel="Werkelijk")
ax.grid(False)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_val, baseline_pred).ravel()
own = metrics_from_confusion(tn, fp, fn, tp)
sklearn_values = {
    "accuracy": accuracy_score(y_val, baseline_pred),
    "precision": precision_score(y_val, baseline_pred, zero_division=0),
    "recall": recall_score(y_val, baseline_pred),
    "specificity": recall_score(y_val, baseline_pred, pos_label=0),
    "f1": f1_score(y_val, baseline_pred)}
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"ROC-AUC = {roc_auc_score(y_val, baseline_scores):.3f}, "
      f"PR-AUC = {average_precision_score(y_val, baseline_scores):.3f}")
pd.DataFrame({"zelf berekend": own, "scikit-learn": sklearn_values}).round(4)